# tokenguard — stop a runaway loop before it spends the money

A budget that reports overspend after the fact is an invoice, not a control. `budget()` projects each call **before** it runs and refuses the one that would cross the cap.

> **Offline.** No API key, no network — the provider is a fake with the real client's *shape*, or a
> committed cassette. This notebook runs in CI on Python 3.11 and 3.13 via `nbmake`, so if a cell
> below stops working the build goes red.
>
> Beside it, [`main.py`](main.py) is the same story as a script. The last cell here asserts what
> that script asserts.

In [ ]:
# The notebook sits beside the recipe, so its own module is importable. Everything below reuses the
# recipe's fixtures rather than re-inventing them — a notebook that built its own fake could drift
# away from what `main.py` proves and nobody would notice.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

## 1 · The loop that would overspend

The recipe's fake reports a deliberately large usage so the cap has something to bite on. ⚠️ Those are the *fake's* numbers — a real `gpt-4o` reply is ~50 output tokens, not 6,000, so live the same cap lasts far longer (~turn 27, measured).

In [ ]:
import main as recipe

print(f"fake usage per call: {recipe.IN_TOKENS:,} in / {recipe.OUT_TOKENS:,} out")

## 2 · Run it under a $0.50 cap

The block is an exception, not a log line.

In [ ]:
from cendor.core import instrument
from cendor.tokenguard import BudgetExceeded, report, reset

reset()
client = instrument(recipe.fake_openai())
try:
    recipe.run_agent_loop(client)
except BudgetExceeded as e:
    print(f"{type(e).__name__}: {e}")

## 3 · Which turns actually ran

`report()` reads tokenguard's own records — not a total this notebook kept.

In [ ]:
r = report(group_by=["feature"])
for row in sorted(r, key=lambda x: x["tags"].get("feature", "")):
    print(f"  {row['tags']['feature']:<11} {row['calls']} calls   ${row['usd'].amount}")
print(f"  {'TOTAL':<11} {sum(row['calls'] for row in r)} calls   ${r.total().amount}")

## 4 · Prove it

In [ ]:
ran = sum(row["calls"] for row in r)
assert ran == 5, f"the $0.50 cap should let 5 turns through and block the 6th, got {ran}"
assert r.total().amount > 0
print("OK — the 6th turn was blocked pre-flight, $0 spent on it")